# Semaine 10 — Jour 6 : Embeddings spécialisés par apprentissage métrique

Apprendre réellement une projection supervisée sur les seuls tickets train.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour l’unité complète avec entretien et challenge. L’installation de l’environnement ajoute un temps distinct; le lab LoRA est optionnel.

Prérequis : Python 3.11+, bases de classification et de vecteurs. Le parcours utilise NumPy et scikit-learn, sans modèle à télécharger ni clé API. Installer les dépendances S10 selon [le guide](../../../docs/week10/QUICKSTART.md).

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Tenter [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks depuis la racine : `notebooks/week10/S10_J6_embeddings_specialises.ipynb` et `notebooks/week10/teacher/S10_J6_embeddings_specialises_teacher.ipynb`.

Critère de réussite : Interpréter une pertinence par catégorie comme un proxy limité.

Les apprentissages du parcours principal sont réellement exécutables. Le corpus reste fictif, les baselines à règles sont identifiées et aucun résultat LoRA n’est annoncé.


# Objectifs mesurables

- Apprendre réellement une projection supervisée sur les seuls tickets train.
- Comparer un espace TF-IDF et sa projection NCA avec un index documentaire train.
- Interpréter une pertinence par catégorie comme un proxy limité.

## Preuves attendues

Une sortie reproductible, un invariant vérifié et une limite de la tâche correctement interprétée.


# S10 J6 — Embeddings spécialisés par apprentissage métrique


## 1. De la représentation générale à une tâche
S9 présentait espaces vectoriels et compatibilité d’index. Ce jour apprend une transformation à partir d’étiquettes : les tickets d’une même catégorie doivent devenir utiles comme voisins. Il ne réécrit pas le moteur de retrieval de S9 et ne présente pas un index supervisé comme une compréhension sémantique générale.

Le point de départ est le TF-IDF appris seulement sur train. Neighborhood Components Analysis apprend une projection linéaire supervisée, optimisée pour une décision par voisinage. Le module utilise une représentation dense du tout petit corpus; ce choix ne serait pas adapté sans mesure à un très grand vocabulaire.

## 2. Apprendre sans voir les questions test
La projection reçoit les 36 tickets train et leurs étiquettes. Les douze tickets test sont des requêtes après entraînement; l’index de voisins contient uniquement les tickets train. Le test n’est pas un index dans lequel on retrouverait la question elle-même.

La dimension cible est huit et la graine est fixée. Le rapport garde le nombre d’itérations et les éventuels avertissements d’optimisation. Le code borne les threads de calcul pour limiter la variabilité de cette petite expérience; il ne promet pas des poids identiques sur toutes les versions matérielles et logicielles.

## 3. Ce que la projection peut apprendre
NCA cherche une géométrie utile aux catégories connues. Elle peut rapprocher des tickets de déploiement et éloigner des refus d’accès, même si certains mots se recouvrent. Elle peut aussi écraser des distinctions importantes à l’intérieur d’une catégorie : deux incidents performance n’ont pas forcément le même remède.

La pertinence du rapport est donc un proxy « même catégorie », pas une annotation humaine de réponse à une question. Un meilleur classement par thème ne prouve pas qu’un ticket explique le problème de l’autre. Distinguer compression, classification par voisinage et recherche de connaissance réellement pertinente.

## 4. Protocole de comparaison
Les deux modes utilisent le même index train et les mêmes requêtes test. On calcule le cosinus après normalisation dans chaque espace. Le top trois est départagé de manière stable par identifiant lorsque les scores sont égaux. Les requêtes sont ensuite évaluées selon la catégorie des voisins.

Precision@3 mesure la part des trois voisins du bon thème. MRR@3 mesure le rang du premier. Recall@3 divise par tous les documents train du thème : douze par catégorie. Son maximum dans ce corpus est donc 3/12=0,25, même avec trois voisins corrects. Ne pas interpréter ce plafond comme un défaut du calcul.

## 5. Lire un gain sans l’exagérer
Dans l’environnement de référence, l’espace passe de 441 dimensions TF-IDF à huit dimensions NCA. La précision thématique moyenne des trois premiers voisins augmente dans cette expérience. Le premier pertinent était déjà bien placé, donc MRR peut rester inchangé. Le rapport conserve les cas et les deux modes au lieu d’un seul chiffre favorable.

Un corpus différent, d’autres catégories ou une distribution modifiée peuvent inverser ce résultat. Pour une vraie spécialisation sémantique, annoter des paires pertinentes et non pertinentes, choisir une méthode adaptée et réserver un test hors des templates de développement. Le module n’entraîne pas d’encodeur neuronal ni de Sentence Transformer.

## 6. Versionner l’espace
La projection dépend du vocabulaire, des IDF, des composants appris et des données. Le rapport conserve une empreinte des caractéristiques et du train. Changer l’un de ces éléments demande de reconstruire les vecteurs de l’index; des dimensions égales ne suffisent pas à assurer la compatibilité.

## À retenir
Un embedding spécialisé optimise une tâche définie. Une bonne géométrie pour les catégories n’est pas automatiquement une bonne géométrie pour toutes les questions métier.


## Diagramme du jour

```mermaid
flowchart TD
 A["TF-IDF fit sur train"] --> B["Matrice train et labels"]
 B --> C["Projection NCA apprise"]
 C --> D["Index train projeté"]
 E["Requêtes test transformées"] --> F["Classement cosinus"]
 D --> F
 F --> G["Métriques par thème"]
```

Le rendu Mermaid dépend du lecteur; la source .mmd reste disponible.

# Lab exécutable

NumPy et scikit-learn sont requis; aucune requête réseau dans les cellules.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week10/nlp.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from examples.week10.nlp import specialized, embedding_report
projection, details = specialized()
report = embedding_report()
assert report["queries"] == 12 and report["index_documents"] == 36
assert report["modes"]["supervised_nca"]["dimension"] == 8
print({mode:{key:value for key,value in metrics.items() if key != "cases"} for mode,metrics in report["modes"].items()})
print(details)

# Lab J6 — Embeddings spécialisés par apprentissage métrique

Depuis la racine du dépôt, dans l’environnement S10 :

```bash
python3 book/week10/day06/labs/run.py
python3 -m examples.week10.demo --day 6
```

Résultat attendu : **projection apprise vers 8 dimensions et métriques de voisinage par thème**.

## Travail

1. Exécuter le scénario et identifier la nature de chaque composant : appris, adapté ou à règles.
2. Réaliser l’exercice 2 sur une copie, sans modifier le corpus distribué.
3. Expliquer le dénominateur du résultat et une limite.

Les données sont sous `examples/week10/data`; le code commun sous `examples/week10/nlp.py`; les tests sous `tests/week10`. Lire [QUICKSTART](../../../../docs/week10/QUICKSTART.md) pour les dépendances, artefacts et commandes.

Les modèles entraînés sont conservés en mémoire dans chaque processus pour accélérer les notebooks. Une modification des données demande un nouveau processus ou de vider explicitement les caches trained/specialized avant de réentraîner. Les correctifs du notebook enseignant s’exécutent après le lab élève.


# Exercices

Conserver une première réponse avant les corrigés.

## Exercice 1 — Calculer le plafond

Douze documents pertinents par thème et k=3 : quel Recall@3 maximal ?

Livrable : réponse argumentée avec calcul, sortie ou test.

## Exercice 2 — Tracer les données

Pourquoi l’index de voisins n’inclut-il pas les tickets test ?

Livrable : réponse argumentée avec calcul, sortie ou test.

## Exercice 3 — Interpréter la catégorie

Pourquoi deux incidents de même classe peuvent-ils être de mauvais voisins métier ?

Livrable : réponse argumentée avec calcul, sortie ou test.


# Entretien

Répondre en deux minutes par question avec un exemple et une limite.

## Question 1

Que prouve une réduction de dimension ?

## Question 2

Quand reconstruire les embeddings ?


# Challenge

Comparer Precision@3, Recall@3 et MRR@3 pour TF-IDF et NCA, puis inspecter une requête dont les voisins changent. Expliquer ce que le gain thématique ne permet pas de conclure.

## Critères

Protocole reproductible, invariant explicite et interprétation honnête du résultat, y compris sans amélioration.


# Références — Jour 6

Sources primaires consultées le 20 octobre 2026. Le corpus et les annotations du lab sont fictifs; les seuils sont pédagogiques.

- [scikit-learn 1.8 : pièges et fuite de données](https://scikit-learn.org/1.8/common_pitfalls.html).
- [SGDClassifier 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.linear_model.SGDClassifier.html) — log_loss et partial_fit.
- [TfidfVectorizer 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).
- [Neighborhood Components Analysis 1.8](https://scikit-learn.org/1.8/modules/generated/sklearn.neighbors.NeighborhoodComponentsAnalysis.html) — apprentissage métrique supervisé.
- [spaCy : entités nommées](https://spacy.io/usage/linguistic-features#named-entities) — vocabulaire et conventions; spaCy n’est pas utilisé dans le lab obligatoire.
- [PEFT : LoRA](https://huggingface.co/docs/peft/package_reference/lora) — extension Transformer optionnelle, non exécutée.

Lire en priorité les références du thème étudié. Les concepts communs à plusieurs bibliothèques ne rendent pas leurs objets ou artefacts interchangeables. Les dépendances obligatoires correspondent à l’environnement testé, les dépendances LoRA restent à verrouiller pour l’expérience optionnelle.
